In [1]:
import numpy as np
import pandas as pd
from pipelines import estimate_regression_models_noLASSO
from joblib import Parallel, delayed
from tqdm import tqdm


In [2]:
def compute_errnorm_mix(i, p, n, nn, beta_0, w_0, sigma, sigma1, psi_delta=1.35, psi_eta=0.1, tau_range=None):
    np.random.seed(i) # 为每个并行工作设置独立的种子

    # --- 数据生成 ---
    uniform_values = np.random.uniform(0, np.sqrt(3), n//2)
    X11 = np.random.normal(size=(n//2, p)) * uniform_values[:, np.newaxis]
    Y11 = X11 @ beta_0 + np.random.standard_cauchy(size=n//2)

    X12 = np.random.normal(size=(n//2, p))
    Y12 = X12 @ beta_0 + np.random.normal(0, sigma, n//2)

    X = np.vstack((X11, X12))
    Y = np.concatenate((Y11, Y12))

    uniform_values2 = np.random.uniform(0, np.sqrt(3), nn//2)
    X21 = np.random.normal(size=(nn//2, p)) * uniform_values2[:, np.newaxis]
    Y21 = X21 @ w_0 + np.random.standard_cauchy(size=nn//2) * sigma1

    X22 = np.random.normal(size=(nn//2, p))
    Y22 = X22 @ w_0 + np.random.normal(0, sigma1, nn//2)

    X1 = np.vstack((X21, X22))
    Y1 = np.concatenate((Y21, Y22))

    #####################
    if tau_range is None:
        tau_range = np.logspace(-4, 1, 15) # 更细致的 tau 范围

    # 调用函数
    estimated_models = estimate_regression_models_noLASSO(
        X, Y,
        X1, Y1,
        delta_param=psi_delta,
        eta_param=psi_eta,
        tau_range=tau_range
    )

    beta_sr = estimated_models["single_robust_ridge"]["betahat"]
    beta_tr = estimated_models["transfer_robust_ridge"]["betahat"]
    beta_pr = estimated_models["pooled_robust_ridge"]["betahat"]


    errnorm_sr = np.sum((beta_sr - beta_0)**2) / np.sum(beta_0**2)
    errnorm_tr = np.sum((beta_tr - beta_0)**2) / np.sum(beta_0**2)
    errnorm_pr = np.sum((beta_pr - beta_0)**2) / np.sum(beta_0**2)

    # 提取最优 tau 值
    optimal_tau_sr = estimated_models["single_robust_ridge"]["optimal_tau"]
    optimal_tau_source_tr = estimated_models["transfer_robust_ridge"]["optimal_tau_source"]
    optimal_tau_target_diff_tr = estimated_models["transfer_robust_ridge"]["optimal_tau_target_diff"]
    optimal_tau_pr = estimated_models["pooled_robust_ridge"]["optimal_tau"]

    return (errnorm_sr, errnorm_tr, errnorm_pr,
            optimal_tau_sr, optimal_tau_source_tr, optimal_tau_target_diff_tr, optimal_tau_pr)

In [3]:
def run_simulation_cv(p, n, K, dd, n_jobs=-1, tau_range=None):
    """
    运行完整的模拟研究，使用CV优化tau参数。

    Args:
        p (int): 特征维度
        n (int): 目标任务样本量
        K (int): 模拟重复次数
        dd (float): 控制delta_0范数的系数
        n_jobs (int): 并行使用的CPU核心数
        tau_range (array, optional): 用于CV的tau候选值范围

    Returns:
        tuple: (mean_errnorm, std_errnorm, errnorm_df, ridge_tau_stats)
    """
    # 简化输出，只显示进度条
    # print(f"Starting CV simulation with p={p}, n={n}, K={K}, dd={dd}")

    # --- 在函数内部定义固定的模拟参数 ---
    nn = n * 2          # 源任务样本量
    sigma = 1           # 目标任务噪声标准差
    sigma1 = 2          # 源任务噪声标准差
    psi_delta = 1.35    # psi 函数参数 delta
    psi_eta = 0.1       # psi 函数参数 eta
    kappa = p // n      # 维度样本比

    if tau_range is None:
        tau_range = np.logspace(-3, 3, 10)  # 默认的tau值范围

    # --- 生成固定的真实系数 (在所有 K 次运行中保持不变) ---
    rng = np.random.RandomState(1) # 使用固定的种子以保证 beta_0, w_0 可复现
    beta_0 = rng.uniform(size=p)
    beta_0 /= np.linalg.norm(beta_0, 2)

    # delta_0 = rng.uniform(size=p)
    # delta_0 /= np.linalg.norm(delta_0, 2) * dd
    delta_0 = np.ones(p) * dd / np.sqrt(p)  # 固定的 delta_0

    w_0 = beta_0 - delta_0

    # --- 使用 joblib 进行并行计算 ---
    # 简化输出，只显示进度条
    # print(f"Running {K} simulations in parallel using {n_jobs if n_jobs > 0 else os.cpu_count()} cores...")
    results_list = Parallel(n_jobs=n_jobs)(
        delayed(compute_errnorm_mix)(
            i, p, n, nn, beta_0, w_0, sigma, sigma1, psi_delta, psi_eta, tau_range
        )
        for i in tqdm(range(K), desc=f"dd={dd:.3f}")
    )

    # --- 结果处理 ---
    # 分离误差和最优tau值
    errnorm_values = np.array([r[0:3] for r in results_list])  # K x 3 array
    tau_values = np.array([r[3:7] for r in results_list])      # K x 4 array

    mean_errnorm = np.nanmean(errnorm_values, axis=0)
    std_errnorm = np.nanstd(errnorm_values, axis=0)


    errnorm_df = pd.DataFrame(errnorm_values, columns=['Single RR', 'Trans RR', 'Pooled RR'])

    tau_columns = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    all_taus_df = pd.DataFrame(tau_values, columns=tau_columns)

    return mean_errnorm, std_errnorm, errnorm_df, all_taus_df


In [4]:
p_val = 400
n_val = 400
K_val = 1000  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = -1  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(0, 1, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(np.e, np.arange(-2.0, 1.5, 0.5))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:

    mean_err, std_err, results_df, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 计算每个 tau 列的众数和频次
    tau_stats_for_dd = {}
    tau_columns_for_stats = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    for col_name in tau_columns_for_stats:
        if col_name in all_t_df:
            counts = all_t_df[col_name].value_counts()
            if not counts.empty:
                most_frequent_tau = counts.index[0]
                frequency = counts.iloc[0]
                tau_stats_for_dd[col_name] = (most_frequent_tau, frequency)
            else:
                tau_stats_for_dd[col_name] = (np.nan, 0) # 处理空 Series 的情况
        else:
            tau_stats_for_dd[col_name] = (np.nan, 0) # 如果列不存在

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'all_taus_df': all_t_df,
            'tau_stats': tau_stats_for_dd # 新增 tau 统计信息
        }

# 汇总错误结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage:
        for i, method in enumerate(error_method_names):
            mean = all_results_storage[dd_val]['mean_err'][i]
            std = all_results_storage[dd_val]['std_err'][i]
            results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
        print(results_line)
    else:
        print(f"{dd_val:<10.3f} No results found.") # 以防万一


# 汇总 Tau 统计结果
print("\n\n" + "=" * 130)
print("Summary of Most Frequent Tau Values and Their Counts Across Different dd Values".center(130))
print("=" * 130)

# tau_columns_for_stats 已在上面定义
# ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
header_tau = f"{'dd':<10}"
for tau_method_name in tau_columns_for_stats:
    header_tau += f"{tau_method_name + ' (Freq)':<30}" # 调整宽度
print(header_tau)
print("-" * 130)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage and 'tau_stats' in all_results_storage[dd_val]:
        stats = all_results_storage[dd_val]['tau_stats']
        for tau_method_name in tau_columns_for_stats:
            if tau_method_name in stats:
                tau_val, freq = stats[tau_method_name]
                tau_results_line += f"{tau_val:.4f} ({freq}/{K_val})".ljust(30) # 显示频次和总次数
            else:
                tau_results_line += "N/A".ljust(30)
        print(tau_results_line)
    else:
        print(f"{dd_val:<10.3f} No tau stats found.")

print("\nSimulation complete.")

dd=2.718: 100%|██████████| 1000/1000 [07:32<00:00,  2.21it/s]




                   Summary of CV Results Across Different dd Values - Mean Error (Std Dev)                    
dd        Single RR                     Trans RR                      Pooled RR                     
--------------------------------------------------------------------------------------------------------------
0.135     0.7738 (0.0383)               0.6739 (0.0381)               0.7033 (0.0428)               
0.223     0.7738 (0.0383)               0.6867 (0.0374)               0.7263 (0.0436)               
0.368     0.7738 (0.0383)               0.7058 (0.0374)               0.7661 (0.0430)               
0.607     0.7738 (0.0383)               0.7365 (0.0388)               0.8288 (0.0368)               
1.000     0.7738 (0.0383)               0.7905 (0.0440)               0.9067 (0.0223)               
1.649     0.7738 (0.0383)               0.9440 (0.0985)               1.0019 (0.0244)               
2.718     0.7738 (0.0383)               1.4854 (0.1076)              

In [5]:
import json

filename = f"res/1mix_cv_results_p{p_val}_simu{K_val}.json"

converted_results = {}
for dd_val in all_results_storage:
    converted_results[str(dd_val)] = {
        'mean_err': all_results_storage[dd_val]['mean_err'].tolist(),
        'std_err': all_results_storage[dd_val]['std_err'].tolist(),
        'results_df': all_results_storage[dd_val]['results_df'].to_dict()
    }

with open(filename, 'w') as f:
    json.dump(converted_results, f, indent=4)

print(f"结果已保存至: {filename}")

结果已保存至: res/1mix_cv_results_p400_simu1000.json


In [ ]:
import json

filename = f"res/mix_cv_results_p{p_val}_simu{K_val}.json"

converted_results = {}
for dd_val in all_results_storage:
    converted_results[str(dd_val)] = {
        'mean_err': all_results_storage[dd_val]['mean_err'].tolist(),
        'std_err': all_results_storage[dd_val]['std_err'].tolist(),
        'results_df': all_results_storage[dd_val]['results_df'].to_dict()
    }

with open(filename, 'w') as f:
    json.dump(converted_results, f, indent=4)

print(f"结果已保存至: {filename}")

In [4]:
p_val = 400
n_val = 400
K_val = 500  # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = -1  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(0, 1, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(np.e, np.arange(-0.5, 2.5, 0.5))

# 存储所有运行的结果
all_results_storage = {}

for dd_val in dd_values:

    mean_err, std_err, results_df, all_t_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 计算每个 tau 列的众数和频次
    tau_stats_for_dd = {}
    tau_columns_for_stats = ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
    for col_name in tau_columns_for_stats:
        if col_name in all_t_df:
            counts = all_t_df[col_name].value_counts()
            if not counts.empty:
                most_frequent_tau = counts.index[0]
                frequency = counts.iloc[0]
                tau_stats_for_dd[col_name] = (most_frequent_tau, frequency)
            else:
                tau_stats_for_dd[col_name] = (np.nan, 0) # 处理空 Series 的情况
        else:
            tau_stats_for_dd[col_name] = (np.nan, 0) # 如果列不存在

    # 存储结果
    all_results_storage[dd_val] = {
            'mean_err': mean_err,
            'std_err': std_err,
            'results_df': results_df,
            'all_taus_df': all_t_df,
            'tau_stats': tau_stats_for_dd # 新增 tau 统计信息
        }

# 汇总错误结果
error_method_names = ['Single RR', 'Trans RR', 'Pooled RR']
print("\n\n" + "=" * 110)
print("Summary of CV Results Across Different dd Values - Mean Error (Std Dev)".center(110))
print("=" * 110)

header_err = f"{'dd':<10}"
for method in error_method_names:
    header_err += f"{method:<30}"
print(header_err)
print("-" * 110)

for dd_val in dd_values:
    results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage:
        for i, method in enumerate(error_method_names):
            mean = all_results_storage[dd_val]['mean_err'][i]
            std = all_results_storage[dd_val]['std_err'][i]
            results_line += f"{mean:.4f} ({std:.4f})".ljust(30)
        print(results_line)
    else:
        print(f"{dd_val:<10.3f} No results found.") # 以防万一


# 汇总 Tau 统计结果
print("\n\n" + "=" * 130)
print("Summary of Most Frequent Tau Values and Their Counts Across Different dd Values".center(130))
print("=" * 130)

# tau_columns_for_stats 已在上面定义
# ['Tau SR', 'Tau TR (Source)', 'Tau TR (Target Diff)', 'Tau PR']
header_tau = f"{'dd':<10}"
for tau_method_name in tau_columns_for_stats:
    header_tau += f"{tau_method_name + ' (Freq)':<30}" # 调整宽度
print(header_tau)
print("-" * 130)

for dd_val in dd_values:
    tau_results_line = f"{dd_val:<10.3f}"
    if dd_val in all_results_storage and 'tau_stats' in all_results_storage[dd_val]:
        stats = all_results_storage[dd_val]['tau_stats']
        for tau_method_name in tau_columns_for_stats:
            if tau_method_name in stats:
                tau_val, freq = stats[tau_method_name]
                tau_results_line += f"{tau_val:.4f} ({freq}/{K_val})".ljust(30) # 显示频次和总次数
            else:
                tau_results_line += "N/A".ljust(30)
        print(tau_results_line)
    else:
        print(f"{dd_val:<10.3f} No tau stats found.")

print("\nSimulation complete.")

dd=7.389: 100%|██████████| 500/500 [03:38<00:00,  2.29it/s]




                   Summary of CV Results Across Different dd Values - Mean Error (Std Dev)                    
dd        Single RR                     Trans RR                      Pooled RR                     
--------------------------------------------------------------------------------------------------------------
0.607     0.7738 (0.0379)               0.9682 (0.1020)               0.9809 (0.0325)               
1.000     0.7738 (0.0379)               0.7809 (0.0469)               0.8786 (0.0268)               
1.649     0.7738 (0.0379)               0.7278 (0.0389)               0.8033 (0.0376)               
2.718     0.7738 (0.0379)               0.7002 (0.0374)               0.7497 (0.0414)               
4.482     0.7738 (0.0379)               0.6826 (0.0369)               0.7163 (0.0413)               
7.389     0.7738 (0.0379)               0.6714 (0.0374)               0.6969 (0.0405)               


                         Summary of Most Frequent Tau Values and Th

In [5]:
import json

filename = f"res/mix_cv_results_p{p_val}_simu{K_val}.json"

converted_results = {}
for dd_val in all_results_storage:
    converted_results[str(dd_val)] = {
        'mean_err': all_results_storage[dd_val]['mean_err'].tolist(),
        'std_err': all_results_storage[dd_val]['std_err'].tolist(),
        'results_df': all_results_storage[dd_val]['results_df'].to_dict()
    }

with open(filename, 'w') as f:
    json.dump(converted_results, f, indent=4)

print(f"结果已保存至: {filename}")

结果已保存至: res/mix_cv_results_p400_simu500.json


In [8]:
p_val = 200
n_val = 200
K_val = 50    # 由于CV会显著增加计算时间，可以减少重复次数
n_jobs_val = -1  # 使用所有 CPU 核心

# print(f"Running CV simulations with p={p_val}, n={n_val}, K={K_val}")

# 设置CV的tau范围
tau_range_val = np.logspace(-2, 1, 10, base=3)

# 要测试的 dd 值
dd_values = np.power(2.0, np.arange(-2, 3.5, 0.5))

# 存储所有运行的结果
all_results = {}

for dd_val in dd_values:
    # 简化输出，移除分隔线
    # print(f"\n\n{'=' * 50}")
    # print(f"Running CV simulation with dd = {dd_val}")
    # print(f"{'=' * 50}")

    # 运行模拟
    mean_err, std_err, results_df = run_simulation_cv(
        p=p_val, n=n_val, K=K_val, dd=dd_val, n_jobs=n_jobs_val, tau_range=tau_range_val
    )

    # 存储结果
    all_results[dd_val] = {
        'mean_err': mean_err,
        'std_err': std_err,
        'results_df': results_df
    }


# 汇总结果
method_names = ['Single RR', 'Trans RR', 'Pooled RR']

print("\n\n" + "=" * 100)
print("Summary of CV Results Across Different dd Values - Mean (Std Dev)".center(100))
print("=" * 100)

# 打印表头
header = f"{'dd':<8}"
for method in method_names:
    header += f"{method:<25}"
print(header)
print("-" * 100)

# 按照dd值组织结果，合并均值和标准差
for dd_val in dd_values:
    results_line = f"{dd_val:.3f}".ljust(8)
    for i, method in enumerate(method_names):
        mean = all_results[dd_val]['mean_err'][i]
        std = all_results[dd_val]['std_err'][i]
        results_line += f"{mean:.4f} ({std:.4f})".ljust(25)
    print(results_line)

dd=8.000: 100%|██████████| 50/50 [00:37<00:00,  1.35it/s]




                 Summary of CV Results Across Different dd Values - Mean (Std Dev)                  
dd      Single RR                Trans RR                 Pooled RR                
----------------------------------------------------------------------------------------------------
0.250   1.2865 (1.7411)          5.3285 (2.2784)          4.3563 (3.0045)          
0.354   1.2865 (1.7411)          2.9174 (1.8459)          2.3486 (1.6823)          
0.500   1.2865 (1.7411)          1.9057 (1.7998)          1.6155 (0.9691)          
0.707   1.2865 (1.7411)          1.5610 (1.7947)          1.2790 (0.6203)          
1.000   1.2865 (1.7411)          1.3919 (1.7700)          1.1038 (0.5052)          
1.414   1.2865 (1.7411)          1.3467 (1.7711)          1.0407 (0.4672)          
2.000   1.2865 (1.7411)          1.3170 (1.7732)          0.9806 (0.4419)          
2.828   1.2865 (1.7411)          1.2988 (1.7752)          0.9433 (0.4311)          
4.000   1.2865 (1.7411)          1.2778 